# Differentiable Quantum ODE Solver — State-Vector Validation

This notebook imports the packaged solver from `pyqpanda_alg.DQC`, runs three first-order differential-equation examples, and compares the trained state-vector circuit with an independent classical trajectory.

The training loss uses only the equation residual and the initial value. Classical solutions are evaluated after training and are not optimization labels.

## Circuit model and differentiable residual

For an output component, the circuit expectation is $f_\theta(x)$. The floating initial-value construction is

$$\widehat u_\theta(x)=u_0+f_\theta(x)-f_\theta(x_0).$$

The input uses the Chebyshev tower $\phi_q(x)=2(q+1)\arccos(x)$, followed by `RZ-RX-RZ` rotations and a nearest-neighbour CNOT chain. The observable is the total $Z$ magnetization. The coordinate derivative is obtained by the RY parameter-shift rule,

$$\partial_x f_\theta(x)=\sum_q \frac{\partial\phi_q}{\partial x}\frac{f_\theta(\phi_q+\pi/2)-f_\theta(\phi_q-\pi/2)}{2}.$$

At collocation points $x_j$, the objective is $N^{-1}\sum_j\|R(x_j,\widehat{\boldsymbol u}_\theta,\partial_x\widehat{\boldsymbol u}_\theta)\|_2^2$.

In [ ]:
import sys
from pathlib import Path

# Find the checked-out pyqpanda-algorithm package without hard-coded paths.
candidates = [Path.cwd(), *Path.cwd().parents]
code_root = next((p for p in candidates if (p / 'pyqpanda_alg').is_dir()), None)
if code_root is None:
    raise FileNotFoundError('Open this notebook inside the pyqpanda-algorithm source tree')
if str(code_root) not in sys.path:
    sys.path.insert(0, str(code_root))
from pyqpanda_alg.DQC import ExperimentConfig, make_cases, plot_results, run_experiments
print('Loaded: pyqpanda_alg.DQC')
print(f'Cases: {tuple(make_cases())}')

## Three benchmark equations

The examples cover a rapidly rotating damped mode, a non-diagonal coupled linear system, and a nonlinear Riccati equation with explicit coordinate dependence. The first two have closed-form references; the Riccati reference is generated by an independent fixed-step RK4 integration.

In [ ]:
CONFIG = ExperimentConfig(
    n_qubits=3,
    depth=2,
    collocation_points=24,
    steps=8,
    holdout_points=101,
    seed=17,
)
results = run_experiments('all', config=CONFIG, backend='numpy')

summary = []
for name, result in results.items():
    summary.append(result['metrics'])
summary

## Numerical audit

`aggregate_rmse` and `max_abs_error` compare the DQC prediction with the post-training classical reference. `holdout_residual_rms` evaluates the differential equation on the held-out coordinates. The flag `training_uses_reference` records that the reference was not supplied to the optimizer.

In [ ]:
for row in summary:
    print(
        f"{row['case']:8s}  loss={row['best_training_loss']:.3e}  "
        f"RMSE={row['aggregate_rmse']:.3e}  max|e|={row['max_abs_error']:.3e}  "
        f"residual RMS={row['holdout_residual_rms']:.3e}"
    )
assert all(row['training_uses_reference'] is False for row in summary)

## DQC trajectories and classical references

Solid curves are the classical references and dashed curves are the trained state-vector DQC outputs. The plotting helper is optional; install Matplotlib if the current Python environment does not already provide it.

In [ ]:
plot_results(results)

## References

[1] O. Kyriienko, A. E. Paine, and V. E. Elfving, "Solving nonlinear differential equations with differentiable quantum circuits," *Physical Review A* **103**, 052416 (2021). DOI: [10.1103/PhysRevA.103.052416](https://doi.org/10.1103/PhysRevA.103.052416).

[2] OriginQ, "QPanda3 documentation," [github.com/OriginQ/QPanda3-doc](https://github.com/OriginQ/QPanda3-doc). The optional native backend follows the documented `CPUQVM` and `VQCircuit` interfaces.

[3] A. Kandala *et al.*, "Hardware-efficient variational quantum eigensolver for small molecules and quantum magnets," *Nature* **549**, 242–246 (2017). DOI: [10.1038/nature23879](https://doi.org/10.1038/nature23879).